# Run RAG on content vector db

## Imports deps

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
from dotenv import load_dotenv
import os

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

In [4]:
CONTENT_COLLECTION_NAME="Content-collection-00"

## Retrieve data

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

In [6]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=CONTENT_COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []  
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["id"])
        retrieved_context.append(result.payload["content_text"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }

In [7]:
results = retrieve_data("Write me a post about decentralized education", qdrant_client, k=20)
results

{'retrieved_context_ids': ['1983186809183178990',
  '1541372971926839299',
  '1500846068204023813',
  '1470324162552074245',
  '9e922d10-a730-43e9-afa4-75be6c537a31',
  '1464251393892667397',
  '1582736606334980097',
  '1960641672611590160',
  '1575024202079686656',
  '1961019767315165204',
  'c2392882-d0b3-4060-a153-7639ac6dafaf',
  '1583400169194033152',
  '1975859495520620689',
  '1968268293883699519',
  '1583048007238361088',
  '1591079193865256960',
  '1962440236861993285',
  '1973732708770836970',
  '1531276504755224576',
  '1965058887532613763'],
 'retrieved_context': ['Currently looking for blockchain student associations for @theguilddotdev \nFound @Blockchain_at_X @42blockchain  @ttu_waa  @KRYPTOSPHERE_ \nLet me know if you have other suggestions!\n#education #Web3Community',
  'I will start tweeting regularly ideas of articles that I want to write and the most liked I will write first',
  'RT @lemiscate: now that noise is tuning down as most portfolios valuation.\n\nThinking

In [8]:
def process_context(context):
    formatted_context=""
    for id,chunk in zip(context["retrieved_context_ids"],context["retrieved_context"]):
        formatted_context+=f"- {id}: {chunk}\n"
    
    return formatted_context

print(process_context(results))
preprocessed_context=process_context(results)

- 1983186809183178990: Currently looking for blockchain student associations for @theguilddotdev 
Found @Blockchain_at_X @42blockchain  @ttu_waa  @KRYPTOSPHERE_ 
Let me know if you have other suggestions!
#education #Web3Community
- 1541372971926839299: I will start tweeting regularly ideas of articles that I want to write and the most liked I will write first
- 1500846068204023813: RT @lemiscate: now that noise is tuning down as most portfolios valuation.

Thinking about making an education thread series about ecosyste…
- 1470324162552074245: It's hard to find good articles vulgarizing web3.
Main take:
Web1 is the era of read
Web2 is the era of read and write
Web3 is the era of read, write and own

@drckangelo #web3

https://t.co/N8VHFsJ30r
- 9e922d10-a730-43e9-afa4-75be6c537a31: Article title: The Guild — an introduction to a peer-run organization for developers
 Article section: Why the Guild matters
 Article content: Decentralized technologies remind us that power can flow from com

## Rest of RAG piepeline

### Prompt

In [9]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
You are an automated Bluesky account for The Guild.

The Guild is a peer‑run organization for software developers. We learn together, certify each other’s skills, and create opportunities through collaboration, attestations, and on‑chain credentials.

Why it matters:
- Community‑verified skills: members issue attestations that build portable, credible profiles.
- Learning by doing: contribute, earn badges, and grow through real projects.
- Open and merit‑based: progress is transparent and anchored on public infrastructure.

As a social media marketing specialist, your job is to create engaging and authentic posts for The Guild's Bluesky feed. 
Generate creative and informative social posts that highlight Guild values, activities, and member achievements, and help grow interest and participation.

Instructions:
- Base your post only on the provided information about activities, topics, and resources.
- Refer to “The Guild”, “Guild members”, projects, attestations, or accomplishments as appropriate for a real community post.
- The post should be captivating, social, and concise (target 220 characters including spaces, tags, and links to stay within Bluesky’s 300-character cap).
- Use an insightful or positive tone and encourage engagement or participation. 
- Do NOT sound like a generic bot.

Guild Activities/Topics:
{preprocessed_context}

Prompt:
{question}
"""
    return prompt

### Answer

In [10]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0  # High temperature for more creative output
    )
    return response.choices[0].message.content

### RAG Pipeline

In [11]:
def rag_pipeline(question,top_k=5):
    retrieved_context=retrieve_data(question,qdrant_client,top_k)
    preprocessed_context=process_context(retrieved_context)
    print(preprocessed_context)
    prompt=build_prompt(preprocessed_context,question)
    answer=generate_answer(prompt)
    return answer

In [12]:
print(rag_pipeline("Write a post along the lines of at the age of ai its hard to trust anything we read online, this make a great case for on-chain reputation systems",top_k=10))

- 1993699573861089526: RT @theguilddotdev: On-chain reputation for developers — peer badges, attestations, and proof of your work.  

Start here → https://t.co/Gh…
- c2392882-d0b3-4060-a153-7639ac6dafaf: Article title: The Guild — an introduction to a peer-run organization for developers
 Article section: Trustful vs trustless systems
 Article content: In the spirit of simplicity, we should not build a trustless certification system that would require a lot of complexity but instead build a trustful network where we give each other badges representing skills that we recognize in each other. The concept is similar to skills on LinkedIn, except it would be on-chain, verifiable, and community-built.
- 1994413250918613177: Agentic systems work best when they use minimal LLM.
LLMs should handle only the nondeterministic steps — the rest should be hard logic.
But in early design, LLMs are great for exploring and finding the shortest path.
#AI #Agents #AgenticAI #DevTools #LLMEngineering
- 14